# Local projections: incertidumbre comercial y producción manufacturera por rama

**Etapa 5 (primera estimación).** Lee la base integrada (`data/processed/base_incertidumbre_comercial_mexico.xlsx`), construye **variaciones a 12 meses** de todas las variables y estima *local projections* en panel con exposición heterogénea.

**Especificación principal** (para cada horizonte h = 0, …, 12 meses):

Δ12 y(i, t+h) = a_i + d_t + β_h · [Δ12 TPU_t × E_i] + γ_h · Δ12 arancel(i, t) + Σ_{l=1..3} [ρ_l Δ12 y(i, t−l) + θ_l Δ12 TPU_{t−l} × E_i + λ_l Δ12 arancel(i, t−l)] + e

- **y:** logaritmo del valor real de la producción (EMIM, miles de pesos de jul-2019). Δ12 y es la variación anual en log.
- **Δ12 TPU:** variación anual del logaritmo del TPU, estandarizada (1 = una desviación estándar).
- **E_i:** exposición de la rama = coeficiente de exportación de la MIP 2018 (decisión D9), entre 0 y 1.
- **a_i, d_t:** efectos fijos de rama y de mes. Los de mes absorben todo lo común a las ramas (ciclo de EE.UU., tipo de cambio, VIX, pandemia como choque común), por eso no se incluyen controles macroeconómicos.
- **Δ12 arancel:** cambio anual de la tasa arancelaria efectiva de EE.UU. a la rama (puntos porcentuales), para separar el arancel aplicado de la incertidumbre.

**β_h** mide cuánto más (o menos) crece en términos anuales la producción de una rama con exposición 1 frente a una con exposición 0, h meses después de un aumento de una desviación estándar en la variación anual del TPU. Se reporta también escalado a la diferencia entre ramas de exposición alta (percentil 75) y baja (percentil 25).

**Errores estándar:** Driscoll-Kraay (robustos a correlación entre ramas y en el tiempo, con rezago máximo 12 + h por el traslape de las variaciones anuales). Como comparación, conglomerados en dos vías (rama y mes).

Este notebook **no** modifica la base integrada: las variaciones se calculan aquí.

In [1]:
import html, platform, warnings
from datetime import datetime
from pathlib import Path
import numpy as np
import pandas as pd
import statsmodels
import statsmodels.api as sm
from IPython.display import HTML, Markdown, display

warnings.filterwarnings("ignore")
def _raiz():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "data" / "processed").is_dir() and (q / "docs").is_dir():
            return q
    raise RuntimeError("No se encontró la raíz del repositorio.")
RAIZ = _raiz()
INICIO = datetime.now()
display(HTML("""<style>
.tabla{font-size:82%;border-collapse:collapse;margin-bottom:8px}.tabla th{background:#dce6f1;border:1px solid #bbb;padding:3px 6px;text-align:left}
.tabla td{border:1px solid #ddd;padding:2px 6px}.irf-grid{display:flex;flex-wrap:wrap;gap:12px}
</style>"""))

def tabla(df, titulo=None, max_filas=80, dec=4):
    if titulo:
        display(Markdown(f"**{titulo}**"))
    d = df.head(max_filas).copy()
    for c in d.columns:
        if pd.api.types.is_float_dtype(d[c]):
            d[c] = d[c].map(lambda v: "NA" if pd.isna(v) else f"{v:,.{dec}f}")
    display(HTML(d.to_html(index=False, border=0, classes="tabla", escape=True)))

def aviso(texto, tipo="info"):
    c = {"info": ("#e8f1fb", "#2c5aa0"), "revisar": ("#fff4e0", "#b26a00"), "ok": ("#e7f5ec", "#1b7f3b")}[tipo]
    display(HTML(f'<div style="background:{c[0]};border-left:6px solid {c[1]};padding:8px 12px;margin:8px 0">{texto}</div>'))

base = pd.read_excel(RAIZ / "data/processed/base_incertidumbre_comercial_mexico.xlsx", sheet_name="base_integrada",
                     dtype={"rama": str, "subsector": str})
base["mes"] = pd.to_datetime(base.mes)
assert not base.duplicated(["rama", "mes"]).any()
display(Markdown(f"**Base:** {base.shape[0]:,} filas, {base.rama.nunique()} ramas, {base.mes.min():%Y-%m} a {base.mes.max():%Y-%m}."))

**Base:** 8,858 filas, 86 ramas, 2018-01 a 2026-07.

## 1. Variaciones a 12 meses

Regla: **diferencia de logaritmos** para variables estrictamente positivas y **diferencia simple** (en puntos) para tasas o índices que pueden valer cero, cuyo logaritmo no existe. La variación se calcula por rama comparando con el mismo mes del año anterior (por fecha, no por posición), así que los primeros 12 meses de cada serie quedan sin variación.

In [2]:
base = base.sort_values(["rama", "mes"]).reset_index(drop=True)
transformaciones = [
    # (variable original, nueva, método, descripción)
    ("emim_valor_produccion_real", "d12_produccion", "log", "Producción real (EMIM)"),
    ("emim_horas_trabajadas", "d12_horas", "log", "Horas trabajadas (EMIM)"),
    ("emim_personal_ocupado", "d12_empleo", "log", "Personal ocupado (EMIM)"),
    ("tpu", "d12_tpu", "log", "TPU (Caldara et al.)"),
    ("wtui_mex", "d12_wtui_mex", "dif", "WTUI México"),
    ("wtui_usa", "d12_wtui_usa", "dif", "WTUI EE.UU."),
    ("wtui_mundo_pib", "d12_wtui_mundo", "dif", "WTUI mundo (ponderado por PIB)"),
    ("epu_mex", "d12_epu", "log", "EPU México"),
    ("eeuu_imp_mx_tasa_efectiva", "d12_arancel", "dif_pp", "Tasa arancelaria efectiva de EE.UU. a la rama"),
    ("tc_fix", "d12_tc", "log", "Tipo de cambio FIX"),
    ("vix_promedio_mensual", "d12_vix", "log", "VIX (promedio mensual)"),
]
anterior = base[["rama", "mes"]].assign(mes=base.mes + pd.DateOffset(years=1))
filas = []
for orig, nueva, metodo, desc in transformaciones:
    x = base[orig].astype(float)
    if metodo == "log":
        assert (x.dropna() > 0).all(), f"{orig} tiene valores no positivos"
        x = np.log(x)
    elif metodo == "dif_pp":
        x = 100 * x
    previo = anterior.assign(v=x.values).rename(columns={"v": "previo"})
    base = base.merge(previo, on=["rama", "mes"], how="left", validate="one_to_one")
    base[nueva] = x.values - base.pop("previo")
    filas.append(dict(variable=nueva, original=orig, descripcion=desc,
                      metodo={"log": "Δ log (≈ variación % / 100)", "dif": "diferencia en puntos del índice",
                              "dif_pp": "diferencia en puntos porcentuales"}[metodo],
                      observaciones=int(base[nueva].notna().sum()), primer_mes=base.mes[base[nueva].notna()].min().strftime("%Y-%m")))
tabla(pd.DataFrame(filas), "Variables transformadas")

# Choques: estandarizados con la desviación estándar de su serie mensual (una observación por mes)
serie_mes = base.drop_duplicates("mes").set_index("mes")
ESCALA = {}
for c in ["d12_tpu", "d12_wtui_mex", "d12_wtui_usa", "d12_wtui_mundo", "d12_epu"]:
    ESCALA[c] = serie_mes[c].std()
    base[c + "_std"] = base[c] / ESCALA[c]
serie_mes = base.drop_duplicates("mes").set_index("mes")
E = base.groupby("rama").mip_coef_exportacion_2018.first()
E_P25, E_P75 = E.quantile(0.25), E.quantile(0.75)
tabla(pd.DataFrame([dict(choque=k, desviacion_estandar=v) for k, v in ESCALA.items()]), "Desviación estándar usada para estandarizar cada choque")
display(Markdown(f"**Exposición (MIP 2018):** percentil 25 = {E_P25:.3f}, percentil 75 = {E_P75:.3f}; diferencia = {E_P75 - E_P25:.3f}."))

**Variables transformadas**

variable,original,descripcion,metodo,observaciones,primer_mes
d12_produccion,emim_valor_produccion_real,Producción real (EMIM),Δ log (≈ variación % / 100),7786,2019-01
d12_horas,emim_horas_trabajadas,Horas trabajadas (EMIM),Δ log (≈ variación % / 100),7826,2019-01
d12_empleo,emim_personal_ocupado,Personal ocupado (EMIM),Δ log (≈ variación % / 100),7826,2019-01
d12_tpu,tpu,TPU (Caldara et al.),Δ log (≈ variación % / 100),7826,2019-01
d12_wtui_mex,wtui_mex,WTUI México,diferencia en puntos del índice,7826,2019-01
d12_wtui_usa,wtui_usa,WTUI EE.UU.,diferencia en puntos del índice,7826,2019-01
d12_wtui_mundo,wtui_mundo_pib,WTUI mundo (ponderado por PIB),diferencia en puntos del índice,7826,2019-01
d12_epu,epu_mex,EPU México,Δ log (≈ variación % / 100),7826,2019-01
d12_arancel,eeuu_imp_mx_tasa_efectiva,Tasa arancelaria efectiva de EE.UU. a la rama,diferencia en puntos porcentuales,7735,2019-01
d12_tc,tc_fix,Tipo de cambio FIX,Δ log (≈ variación % / 100),7826,2019-01


**Desviación estándar usada para estandarizar cada choque**

choque,desviacion_estandar
d12_tpu,0.8552
d12_wtui_mex,0.2833
d12_wtui_usa,0.2628
d12_wtui_mundo,"13,192.9056"
d12_epu,0.6950


**Exposición (MIP 2018):** percentil 25 = 0.158, percentil 75 = 0.618; diferencia = 0.460.

### La variación anual del TPU en el periodo de estimación
El efecto se identifica con pocos episodios: la caída de 2020-2021 (que coincide con la pandemia) y el alza de 2024-2025.

In [3]:
def svg_linea(x, y, titulo, ylab, ancho=860, alto=230, sombra=None):
    L, R, T, B = 60, 20, 30, 38
    xs = np.arange(len(x))
    ymin, ymax = np.nanmin(y), np.nanmax(y)
    pad = 0.08 * (ymax - ymin)
    ymin, ymax = ymin - pad, ymax + pad
    X = lambda i: L + i / (len(x) - 1) * (ancho - L - R)
    Y = lambda v: T + (ymax - v) / (ymax - ymin) * (alto - T - B)
    p = [f'<svg viewBox="0 0 {ancho} {alto}" style="width:100%;max-width:{ancho}px;font:11px system-ui,sans-serif" role="img" aria-label="{html.escape(titulo)}">',
         f'<text x="{L}" y="16" style="font-size:13px;font-weight:600;fill:#0b0b0b">{html.escape(titulo)}</text>']
    if sombra is not None:
        for i0, i1 in sombra:
            p.append(f'<rect x="{X(i0):.1f}" y="{T}" width="{X(i1) - X(i0):.1f}" height="{alto - T - B}" fill="#e6e5e1"/>')
    for t in np.linspace(ymin, ymax, 5):
        p.append(f'<line x1="{L}" x2="{ancho - R}" y1="{Y(t):.1f}" y2="{Y(t):.1f}" stroke="#eeeeec"/><text x="{L - 6}" y="{Y(t) + 4:.1f}" text-anchor="end" fill="#7a7974">{t:.1f}</text>')
    if ymin < 0 < ymax:
        p.append(f'<line x1="{L}" x2="{ancho - R}" y1="{Y(0):.1f}" y2="{Y(0):.1f}" stroke="#52514e" stroke-width="1"/>')
    pts = " ".join(f"{X(i):.1f},{Y(v):.1f}" for i, v in zip(xs, y) if not np.isnan(v))
    p.append(f'<polyline points="{pts}" fill="none" stroke="#2a78d6" stroke-width="2"/>')
    for i, (xx, v) in enumerate(zip(x, y)):
        if not np.isnan(v):
            p.append(f'<circle cx="{X(i):.1f}" cy="{Y(v):.1f}" r="6" fill="transparent"><title>{xx}: {v:.2f}</title></circle>')
    for i, xx in enumerate(x):
        if xx.endswith("-01"):
            p.append(f'<text x="{X(i):.1f}" y="{alto - B + 16}" text-anchor="middle" fill="#7a7974">{xx[:4]}</text>')
    p.append(f'<text x="14" y="{T + (alto - T - B) / 2}" transform="rotate(-90 14 {T + (alto - T - B) / 2})" text-anchor="middle" fill="#7a7974">{html.escape(ylab)}</text></svg>')
    return "".join(p)

s = serie_mes.d12_tpu_std.dropna()
meses_txt = [m.strftime("%Y-%m") for m in s.index]
idx_pand = [i for i, m in enumerate(s.index) if pd.Timestamp("2020-03-01") <= m <= pd.Timestamp("2021-06-01")]
display(HTML(svg_linea(meses_txt, s.values, "Choque: variación anual del log del TPU, estandarizada (gris: pandemia, 2020-03 a 2021-06)",
                       "desviaciones estándar", sombra=[(idx_pand[0], idx_pand[-1])])))

## 2. Estimador
Para cada horizonte se construye Δ12 y(i, t+h), se eliminan los efectos fijos de rama y de mes por proyecciones alternadas (panel no balanceado por faltantes) y se estima por MCO. Los errores estándar Driscoll-Kraay se calculan con `statsmodels` (`nw-groupsum`, agrupando por mes).

In [4]:
PANDEMIA = (pd.Timestamp("2020-03-01"), pd.Timestamp("2021-06-01"))

def demean_two_way(M, g1, g2, tol=1e-10, max_iter=500):
    M = M.copy()
    for _ in range(max_iter):
        antes = M.copy()
        M = M - M.groupby(g1).transform("mean")
        M = M - M.groupby(g2).transform("mean")
        if np.abs(M.values - antes.values).max() < tol:
            break
    return M

def estimar_lp(y="d12_produccion", choque="d12_tpu_std", H=12, lags=3, arancel=True, efectos_tiempo=True,
               excluir_pandemia=False, controles_macro=(), interactuar=True):
    d = base[["rama", "mes", y, choque, "d12_arancel", "mip_coef_exportacion_2018", *controles_macro]].copy()
    g = d.groupby("rama")
    nombre_choque = "choque_x_E" if interactuar else "choque"
    d[nombre_choque] = d[choque] * (d.mip_coef_exportacion_2018 if interactuar else 1.0)
    regs = [nombre_choque]
    for l in range(1, lags + 1):
        d[f"y_l{l}"] = g[y].shift(l)
        d[f"{nombre_choque}_l{l}"] = g[nombre_choque].shift(l)
        regs += [f"y_l{l}", f"{nombre_choque}_l{l}"]
    if arancel:
        d["arancel"] = d["d12_arancel"]
        regs.append("arancel")
        for l in range(1, lags + 1):
            d[f"arancel_l{l}"] = g["d12_arancel"].shift(l)
            regs.append(f"arancel_l{l}")
    for c in controles_macro:
        regs.append(c)
        for l in range(1, lags + 1):
            d[f"{c}_l{l}"] = g[c].shift(l)
            regs.append(f"{c}_l{l}")
    filas = []
    for h in range(H + 1):
        d["y_h"] = g[y].shift(-h)
        d["mes_h"] = g["mes"].shift(-h)
        m = d.dropna(subset=["y_h", *regs]).copy()
        if excluir_pandemia:
            dentro = lambda s: (s >= PANDEMIA[0]) & (s <= PANDEMIA[1])
            m = m[~dentro(m.mes) & ~dentro(m.mes_h)]
        cols = ["y_h", *regs]
        if efectos_tiempo:
            Z = demean_two_way(m[cols], m.rama, m.mes)
            X = Z[regs]
        else:
            Z = m[cols] - m.groupby("rama")[cols].transform("mean")
            X = Z[regs]
        t_idx = m.mes.rank(method="dense").astype(int).values
        mod = sm.OLS(Z["y_h"].values, X.values)
        dk = mod.fit(cov_type="nw-groupsum", cov_kwds={"time": t_idx, "maxlags": 12 + h})
        cl = mod.fit(cov_type="cluster", cov_kwds={"groups": np.column_stack([m.rama.astype("category").cat.codes.values, t_idx])})
        b, se_dk, se_cl = dk.params[0], dk.bse[0], cl.bse[0]
        filas.append(dict(h=h, beta=b, se_dk=se_dk, se_cluster2=se_cl, t_dk=b / se_dk, n=len(m),
                          ramas=m.rama.nunique(), meses=m.mes.nunique(),
                          desde=m.mes.min().strftime("%Y-%m"), hasta=m.mes.max().strftime("%Y-%m")))
    return pd.DataFrame(filas)

# Comprobación del estimador: sin efectos de tiempo ni rezagos, el coeficiente coincide con MCO con dummies
chk = base[["rama", "mes", "d12_produccion", "d12_tpu_std"]].dropna()
dummies = pd.get_dummies(chk.rama, drop_first=True, dtype=float)
b_dummies = sm.OLS(chk.d12_produccion, sm.add_constant(pd.concat([chk.d12_tpu_std, dummies], axis=1))).fit().params["d12_tpu_std"]
Zc = chk[["d12_produccion", "d12_tpu_std"]] - chk.groupby("rama")[["d12_produccion", "d12_tpu_std"]].transform("mean")
b_within = sm.OLS(Zc.d12_produccion, Zc[["d12_tpu_std"]]).fit().params.iloc[0]
assert abs(b_dummies - b_within) < 1e-10
aviso(f"Comprobación del estimador: transformación within = MCO con variables dicotómicas de rama (diferencia {abs(b_dummies - b_within):.1e}).", "ok")

## 3. Resultado principal

In [5]:
def marcas(ymin, ymax, n=5):
    paso = (ymax - ymin) / n
    mag = 10 ** np.floor(np.log10(paso))
    paso = min((m * mag for m in (1, 2, 2.5, 5, 10) if m * mag >= paso), default=10 * mag)
    return np.arange(np.floor(ymin / paso) * paso, ymax + paso / 2, paso), paso

def svg_irf(res, titulo, escala=100.0, ancho=420, alto=250, color="#2a78d6"):
    L, R, T, B = 52, 14, 34, 34
    b = res.beta.values * escala
    lo90, hi90 = (res.beta - 1.645 * res.se_dk).values * escala, (res.beta + 1.645 * res.se_dk).values * escala
    lo95, hi95 = (res.beta - 1.96 * res.se_dk).values * escala, (res.beta + 1.96 * res.se_dk).values * escala
    ticks, paso = marcas(min(lo95.min(), 0), max(hi95.max(), 0))
    ymin, ymax = ticks[0], ticks[-1]
    X = lambda h: L + h / 12 * (ancho - L - R)
    Y = lambda v: T + (ymax - v) / (ymax - ymin) * (alto - T - B)
    banda = lambda lo, hi: " ".join(f"{X(h):.1f},{Y(v):.1f}" for h, v in zip(res.h, hi)) + " " + " ".join(f"{X(h):.1f},{Y(v):.1f}" for h, v in zip(res.h[::-1], lo[::-1]))
    p = [f'<svg viewBox="0 0 {ancho} {alto}" style="width:{ancho}px;max-width:100%;font:11px system-ui,sans-serif" role="img" aria-label="{html.escape(titulo)}">',
         f'<text x="{L}" y="14" style="font-size:12px;font-weight:600;fill:#0b0b0b">{html.escape(titulo)}</text>']
    dec = next(k for k in range(4) if abs(round(paso, k) - paso) < 1e-9)
    for t in ticks:
        p.append(f'<line x1="{L}" x2="{ancho - R}" y1="{Y(t):.1f}" y2="{Y(t):.1f}" stroke="#eeeeec"/><text x="{L - 6}" y="{Y(t) + 4:.1f}" text-anchor="end" fill="#7a7974">{t:.{dec}f}</text>')
    p.append(f'<text x="12" y="{T + (alto - T - B) / 2:.0f}" transform="rotate(-90 12 {T + (alto - T - B) / 2:.0f})" text-anchor="middle" fill="#7a7974">puntos porcentuales</text>')
    p.append(f'<polygon points="{banda(lo95, hi95)}" fill="{color}" fill-opacity="0.12"/>')
    p.append(f'<polygon points="{banda(lo90, hi90)}" fill="{color}" fill-opacity="0.22"/>')
    p.append(f'<line x1="{L}" x2="{ancho - R}" y1="{Y(0):.1f}" y2="{Y(0):.1f}" stroke="#52514e"/>')
    p.append(f'<polyline points="{" ".join(f"{X(h):.1f},{Y(v):.1f}" for h, v in zip(res.h, b))}" fill="none" stroke="{color}" stroke-width="2"/>')
    for h, v, l9, h9 in zip(res.h, b, lo90, hi90):
        p.append(f'<circle cx="{X(h):.1f}" cy="{Y(v):.1f}" r="3" fill="{color}"/><circle cx="{X(h):.1f}" cy="{Y(v):.1f}" r="9" fill="transparent">'
                 f'<title>h={h}: {v:.2f} (IC 90%: {l9:.2f} a {h9:.2f})</title></circle>')
    for h in range(0, 13, 3):
        p.append(f'<text x="{X(h):.1f}" y="{alto - B + 15}" text-anchor="middle" fill="#7a7974">{h}</text>')
    p.append(f'<text x="{(L + ancho - R) / 2}" y="{alto - 4}" text-anchor="middle" fill="#7a7974">meses después del choque (h)</text></svg>')
    return "".join(p)

principal = estimar_lp()
principal_sp = estimar_lp(excluir_pandemia=True)
ESC = 100 * (E_P75 - E_P25)  # puntos porcentuales de crecimiento anual, rama p75 vs p25
display(HTML('<div class="irf-grid">'
             + svg_irf(principal, "Muestra completa", escala=ESC)
             + svg_irf(principal_sp, "Sin pandemia (excluye 2020-03 a 2021-06)", escala=ESC) + "</div>"))
display(Markdown(f"Eje vertical: diferencia en el **crecimiento anual de la producción real (puntos porcentuales)** entre una rama de exposición alta "
                 f"(p75, {E_P75:.2f}) y una de exposición baja (p25, {E_P25:.2f}) ante un aumento de **1 desviación estándar** en la variación anual del TPU. "
                 f"Bandas: intervalos de 90% (oscura) y 95% (clara), errores Driscoll-Kraay."))
for nombre, r in [("Muestra completa", principal), ("Sin pandemia", principal_sp)]:
    t = r.assign(efecto_p75_vs_p25_pp=r.beta * ESC, ic90_inf=(r.beta - 1.645 * r.se_dk) * ESC, ic90_sup=(r.beta + 1.645 * r.se_dk) * ESC)
    tabla(t[["h", "beta", "se_dk", "se_cluster2", "t_dk", "efecto_p75_vs_p25_pp", "ic90_inf", "ic90_sup", "n", "ramas", "meses", "desde", "hasta"]],
          f"{nombre}: coeficientes por horizonte (beta en log; efecto en puntos porcentuales)", dec=3)

Eje vertical: diferencia en el **crecimiento anual de la producción real (puntos porcentuales)** entre una rama de exposición alta (p75, 0.62) y una de exposición baja (p25, 0.16) ante un aumento de **1 desviación estándar** en la variación anual del TPU. Bandas: intervalos de 90% (oscura) y 95% (clara), errores Driscoll-Kraay.

**Muestra completa: coeficientes por horizonte (beta en log; efecto en puntos porcentuales)**

h,beta,se_dk,se_cluster2,t_dk,efecto_p75_vs_p25_pp,ic90_inf,ic90_sup,n,ramas,meses,desde,hasta
0,0.007,0.017,0.017,0.415,0.328,-0.972,1.627,7432,85,88,2019-04,2026-07
1,0.019,0.020,0.021,0.958,0.876,-0.627,2.379,7347,85,87,2019-04,2026-06
2,0.015,0.018,0.018,0.789,0.670,-0.727,2.068,7262,85,86,2019-04,2026-05
3,0.007,0.019,0.025,0.375,0.335,-1.134,1.804,7176,85,85,2019-04,2026-04
4,0.009,0.019,0.023,0.491,0.421,-0.990,1.833,7091,85,84,2019-04,2026-03
5,0.040,0.025,0.026,1.583,1.826,-0.072,3.724,7005,85,83,2019-04,2026-02
6,0.009,0.017,0.018,0.498,0.400,-0.923,1.724,6919,85,82,2019-04,2026-01
7,0.021,0.024,0.021,0.892,0.967,-0.815,2.748,6833,85,81,2019-04,2025-12
8,0.040,0.023,0.025,1.723,1.839,0.083,3.594,6747,85,80,2019-04,2025-11
9,0.015,0.022,0.019,0.649,0.669,-1.026,2.363,6661,85,79,2019-04,2025-10


**Sin pandemia: coeficientes por horizonte (beta en log; efecto en puntos porcentuales)**

h,beta,se_dk,se_cluster2,t_dk,efecto_p75_vs_p25_pp,ic90_inf,ic90_sup,n,ramas,meses,desde,hasta
0,-0.010,0.021,0.020,-0.487,-0.473,-2.073,1.126,6072,85,72,2019-04,2026-07
1,-0.001,0.010,0.020,-0.110,-0.048,-0.772,0.675,5902,85,70,2019-04,2026-06
2,-0.003,0.007,0.017,-0.465,-0.140,-0.634,0.355,5733,85,68,2019-04,2026-05
3,-0.007,0.008,0.020,-0.901,-0.314,-0.888,0.259,5562,85,66,2019-04,2026-04
4,-0.012,0.009,0.019,-1.241,-0.532,-1.237,0.173,5392,85,64,2019-04,2026-03
5,0.008,0.009,0.027,0.959,0.378,-0.271,1.027,5222,85,62,2019-04,2026-02
6,-0.015,0.007,0.015,-2.187,-0.675,-1.184,-0.167,5051,85,60,2019-04,2026-01
7,-0.002,0.018,0.017,-0.105,-0.088,-1.461,1.286,4881,85,58,2019-04,2025-12
8,0.006,0.011,0.013,0.593,0.288,-0.510,1.085,4711,85,56,2019-04,2025-11
9,-0.026,0.012,0.012,-2.062,-1.181,-2.122,-0.239,4541,85,54,2019-04,2025-10


## 4. Robustez
Misma especificación cambiando una cosa a la vez. Todas las gráficas usan la misma escala que la principal (puntos porcentuales, p75 frente a p25, 1 desviación estándar del choque).

In [6]:
robustez = {
    "Horas trabajadas": estimar_lp(y="d12_horas"),
    "Personal ocupado": estimar_lp(y="d12_empleo"),
    "Sin control arancelario": estimar_lp(arancel=False),
    "Horas trabajadas, sin pandemia": estimar_lp(y="d12_horas", excluir_pandemia=True),
    "Choque: WTUI EE.UU.": estimar_lp(choque="d12_wtui_usa_std"),
    "Choque: WTUI mundo": estimar_lp(choque="d12_wtui_mundo_std"),
    "Choque: WTUI México": estimar_lp(choque="d12_wtui_mex_std"),
    "Choque: EPU México": estimar_lp(choque="d12_epu_std"),
}
display(HTML('<div class="irf-grid">' + "".join(svg_irf(r, k, escala=ESC) for k, r in robustez.items()) + "</div>"))
resumen = pd.concat([r.assign(especificacion=k) for k, r in {"Principal: muestra completa": principal,
                                                             "Principal: sin pandemia": principal_sp, **robustez}.items()])
resumen["efecto_p75_vs_p25_pp"] = resumen.beta * ESC
resumen["significativo_90"] = (resumen.t_dk.abs() > 1.645)
tabla(resumen.pivot(index="h", columns="especificacion", values="efecto_p75_vs_p25_pp").reset_index()
      [["h", "Principal: muestra completa", "Principal: sin pandemia", *robustez.keys()]],
      "Efecto en puntos porcentuales por horizonte y especificación", dec=2)
tabla(resumen.groupby("especificacion").agg(horizontes_significativos_90=("significativo_90", "sum"),
                                            efecto_minimo=("efecto_p75_vs_p25_pp", "min"), efecto_maximo=("efecto_p75_vs_p25_pp", "max"),
                                            efecto_h12=("efecto_p75_vs_p25_pp", "last")).reset_index(),
      "Resumen por especificación (de 13 horizontes)", dec=2)

**Efecto en puntos porcentuales por horizonte y especificación**

especificacion,h,Principal: muestra completa,Principal: sin pandemia,Horas trabajadas,Personal ocupado,Sin control arancelario,"Horas trabajadas, sin pandemia",Choque: WTUI EE.UU.,Choque: WTUI mundo,Choque: WTUI México,Choque: EPU México
,0,0.33,-0.47,0.09,0.08,0.26,-0.14,-0.23,-0.34,-1.52,-0.35
,1,0.88,-0.05,0.31,0.40,0.81,0.07,0.14,-0.02,-0.82,0.32
,2,0.67,-0.14,0.14,0.23,0.64,0.03,0.21,0.11,0.78,0.34
,3,0.34,-0.31,-0.09,0.12,0.29,-0.12,1.54,1.19,-1.23,-0.01
,4,0.42,-0.53,-0.23,0.20,0.38,-0.49,0.56,0.55,-1.14,-1.02
,5,1.83,0.38,0.51,0.35,1.81,0.00,-0.71,-0.59,-1.65,0.20
,6,0.40,-0.68,-0.40,0.07,0.42,-0.76,0.23,0.39,-1.93,-0.11
,7,0.97,-0.09,-0.39,-0.05,1.07,-0.82,0.42,0.81,-1.17,0.19
,8,1.84,0.29,-0.21,0.05,1.84,-0.12,-0.17,0.32,-0.51,-0.39
,9,0.67,-1.18,-0.41,-0.15,0.63,-1.27,0.91,1.43,0.62,-0.01


**Resumen por especificación (de 13 horizontes)**

especificacion,horizontes_significativos_90,efecto_minimo,efecto_maximo,efecto_h12
Choque: EPU México,1,-1.02,0.68,0.61
Choque: WTUI EE.UU.,2,-0.71,1.71,1.71
Choque: WTUI México,2,-1.93,3.41,3.41
Choque: WTUI mundo,3,-0.59,2.35,2.35
Horas trabajadas,0,-0.62,0.51,0.02
"Horas trabajadas, sin pandemia",4,-1.27,0.07,-0.51
Personal ocupado,1,-0.25,0.40,-0.25
Principal: muestra completa,2,0.15,1.84,0.15
Principal: sin pandemia,3,-1.18,1.48,0.03
Sin control arancelario,2,0.26,1.84,0.43


## 5. Efecto promedio (preliminar)
Sin efectos fijos de mes: se estima el efecto del choque común sobre todas las ramas (sin interacción con la exposición), con efectos fijos de rama y controles macroeconómicos en variación anual (tipo de cambio y VIX, con 3 rezagos). **Es preliminar:** falta la producción industrial total de EE.UU., que se incorporará aparte; sin ella, el choque puede estar captando la demanda estadounidense. La escala es 1 desviación estándar del choque, en puntos porcentuales del crecimiento anual de la producción.

In [7]:
promedio = estimar_lp(efectos_tiempo=False, interactuar=False, controles_macro=("d12_tc", "d12_vix"))
promedio_sp = estimar_lp(efectos_tiempo=False, interactuar=False, controles_macro=("d12_tc", "d12_vix"), excluir_pandemia=True)
display(HTML('<div class="irf-grid">' + svg_irf(promedio, "Efecto promedio, muestra completa", escala=100)
             + svg_irf(promedio_sp, "Efecto promedio, sin pandemia", escala=100) + "</div>"))
tabla(pd.concat([promedio.assign(muestra="completa"), promedio_sp.assign(muestra="sin pandemia")])
      .assign(efecto_pp=lambda d: 100 * d.beta)[["muestra", "h", "efecto_pp", "se_dk", "t_dk", "n", "meses"]],
      "Efecto promedio por horizonte (puntos porcentuales por 1 desviación estándar)", dec=3)

resumen.to_csv(RAIZ / "outputs/tables/lp_resultados.csv", index=False)
pd.concat([promedio.assign(muestra="completa"), promedio_sp.assign(muestra="sin pandemia")]).to_csv(RAIZ / "outputs/tables/lp_efecto_promedio.csv", index=False)
display(Markdown(f"Resultados guardados en `outputs/tables/lp_resultados.csv` y `outputs/tables/lp_efecto_promedio.csv`.  \n"
                 f"**Fin:** {datetime.now():%Y-%m-%d %H:%M:%S} · Python {platform.python_version()}, pandas {pd.__version__}, statsmodels {statsmodels.__version__}"))

**Efecto promedio por horizonte (puntos porcentuales por 1 desviación estándar)**

muestra,h,efecto_pp,se_dk,t_dk,n,meses
completa,0,-0.070,0.008,-0.085,7432,88
completa,1,0.935,0.013,0.729,7347,87
completa,2,2.218,0.028,0.784,7262,86
completa,3,0.794,0.028,0.285,7176,85
completa,4,1.300,0.031,0.417,7091,84
completa,5,2.364,0.034,0.694,7005,83
completa,6,1.208,0.028,0.428,6919,82
completa,7,1.141,0.023,0.502,6833,81
completa,8,-2.537,0.013,-2.002,6747,80
completa,9,-2.007,0.020,-0.983,6661,79


Resultados guardados en `outputs/tables/lp_resultados.csv` y `outputs/tables/lp_efecto_promedio.csv`.  
**Fin:** 2026-10-01 21:16:25 · Python 3.11.15, pandas 3.0.6, statsmodels 0.15.0